
## Load Preprocessed Dataset


In [1]:

import numpy as np
import pandas as pd
from pathlib import Path

# Locate the project directory
project_root = Path.cwd()

if not (project_root / "data/processed/cleaned_activity_sessions.csv").exists():
    project_root = project_root.parent

# Load the cleaned sensor dataset
data_path = project_root / "data/processed/cleaned_activity_sessions.csv"

df = pd.read_csv(data_path)

# Verify the loaded dataset
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Unique participants:", df["Participant"].nunique())

print(
    "Activity sessions:",
    df[["Task", "Participant", "Session"]]
    .drop_duplicates().shape[0]
)

display(df.head())


Total rows: 286838
Total columns: 16
Unique participants: 38
Activity sessions: 334


,Task,Participant,Session,Time,HR_Processed,HRV,Temperature,IMU_5_ax_g_,IMU_5_ay_g_,IMU_5_az_g_,IMU_5_gx_dps_,IMU_5_gy_dps_,IMU_5_gz_dps_,Physical Fatigue - Initial,Physical Fatigue - Final,Elapsed_Time_s
0,Composite,P1,Rep1,522500,102.5,22.402814,34.224213,0.042525,0.129985,0.011618,14.436704,14.436704,14.436704,1,3,0.0
1,Composite,P1,Rep1,523000,104.7,22.376775,34.226817,-0.061162,-2.334652,0.173983,1.233281,1.233281,1.233281,1,3,0.5
2,Composite,P1,Rep1,523500,108.2,22.351200,34.228885,0.348357,-2.585750,0.150009,13.288418,13.288418,13.288418,1,3,1.0
3,Composite,P1,Rep1,524000,108.2,22.326121,34.230427,0.020237,-0.042484,0.012974,-34.480159,-34.480159,-34.480159,1,3,1.5
4,Composite,P1,Rep1,524500,108.2,22.301569,34.231472,0.039516,-0.311922,0.089224,-5.812640,-5.812640,-5.812640,1,3,2.0


In [3]:
df.shape

(286838, 16)


### Calculate Motion Magnitudes


In [4]:

# Calculate acceleration magnitude from the three ADAM 5 axes
df["Acceleration_Magnitude"] = np.sqrt(
    df["IMU_5_ax_g_"]**2 +
    df["IMU_5_ay_g_"]**2 +
    df["IMU_5_az_g_"]**2
)

# Use one gyroscope axis provisionally because all three axes are duplicated
df["Gyroscope_Proxy"] = df["IMU_5_gx_dps_"].abs()

# View the newly created motion features
display(
    df[
        [
            "Acceleration_Magnitude",
            "Gyroscope_Proxy"
        ]
    ].head(10)
)


,Acceleration_Magnitude,Gyroscope_Proxy
0,0.137257,14.436704
1,2.341925,1.233281
2,2.613419,13.288418
3,0.048813,34.480159
4,0.326829,5.812640
5,0.959307,16.733941
6,0.969873,6.454106
7,0.979266,3.563215
8,1.054481,16.625290
9,0.943052,5.428093



### Extract Session-Level Sensor Features


In [5]:

# Define the five sensor inputs
sensor_features = {
    "HR": "HR_Processed",
    "HRV": "HRV",
    "Temperature": "Temperature",
    "Acceleration": "Acceleration_Magnitude",
    "Gyroscope": "Gyroscope_Proxy"
}

# Select statistical features for each sensor
statistics = ["mean", "median", "std"]

aggregations = {
    f"{sensor}_{stat}": (column, stat)
    for sensor, column in sensor_features.items()
    for stat in statistics
}

# Extract one feature row per activity session
features_df = df.groupby(
    ["Task", "Participant", "Session"],
    as_index=False
).agg(**aggregations)

# Verify extracted features
print("Total ML samples:", len(features_df))
print("Total sensor features:", len(aggregations))

display(features_df.head())


Total ML samples: 334
Total sensor features: 15


,Task,Participant,Session,HR_mean,HR_median,HR_std,HRV_mean,HRV_median,HRV_std,Temperature_mean,Temperature_median,Temperature_std,Acceleration_mean,Acceleration_median,Acceleration_std,Gyroscope_mean,Gyroscope_median,Gyroscope_std
0,Composite,P1,Rep1,130.933060,131.60,5.368690,23.577235,20.575751,10.887614,34.713883,34.681758,0.289609,0.982510,0.981482,0.124014,7.569048,4.819670,7.645788
1,Composite,P1,Rep2,137.085352,138.00,4.807611,12.932734,12.488656,6.593099,35.834101,35.813792,0.290845,0.983967,0.978990,0.111977,7.809359,5.294218,7.535942
2,Composite,P1,Rep3,138.416223,139.75,5.775938,17.844573,14.910803,6.429464,36.789540,36.755521,0.099439,0.986493,0.979032,0.123755,8.464090,6.022367,7.967521
3,Composite,P1,Rep4,139.749239,140.30,3.894888,20.843827,17.715367,12.270275,37.133465,37.119516,0.045261,0.979235,0.975083,0.126438,8.765335,5.985651,8.294293
4,Composite,P1,Rep5,138.760976,139.60,5.279229,13.741781,14.770766,5.037573,37.247084,37.244014,0.017937,0.995320,0.980618,0.153664,9.932089,7.417374,9.636959



### Assign Fatigue Labels   


In [6]:

# Identify each activity session
group_columns = ["Task", "Participant", "Session"]

# Verify that every session has one final fatigue score
label_counts = df.groupby(group_columns)[
    "Physical Fatigue - Final"
].nunique()

assert (label_counts == 1).all(), "Inconsistent fatigue labels found"

# Extract the final fatigue score for each session
fatigue_labels = (
    df.groupby(group_columns, as_index=False)
    ["Physical Fatigue - Final"]
    .first()
    .rename(columns={
        "Physical Fatigue - Final": "Final_Fatigue_Score"
    })
)

# Combine sensor features with their fatigue scores
features_df = features_df.merge(
    fatigue_labels,
    on=group_columns,
    how="left",
    validate="one_to_one"
)

# Convert fatigue scores into three categories
features_df["Fatigue_Level"] = pd.cut(
    features_df["Final_Fatigue_Score"],
    bins=[-0.1, 3, 6, 10],
    labels=["Low", "Moderate", "High"]
)

# Verify the final labels
print("Total ML samples:", len(features_df))

print("\nFatigue Level Distribution:")
print(features_df["Fatigue_Level"].value_counts().sort_index())

display(
    features_df[
        ["Task", "Participant", "Session",
         "Final_Fatigue_Score", "Fatigue_Level"]
    ].head(10)
)


Total ML samples: 334

Fatigue Level Distribution:
Fatigue_Level
Low         102
Moderate    145
High         87
Name: count, dtype: int64


,Task,Participant,Session,Final_Fatigue_Score,Fatigue_Level
0,Composite,P1,Rep1,3,Low
1,Composite,P1,Rep2,5,Moderate
2,Composite,P1,Rep3,6,Moderate
3,Composite,P1,Rep4,7,High
4,Composite,P1,Rep5,7,High
5,Composite,P10,Rep1,1,Low
6,Composite,P10,Rep2,1,Low
7,Composite,P10,Rep3,2,Low
8,Composite,P10,Rep4,3,Low
9,Composite,P10,Rep5,3,Low



### Check Missing Feature Values


In [7]:
# Select all 15 ML features
feature_columns = list(aggregations.keys())

# Count missing values for each feature
missing_counts = features_df[feature_columns].isna().sum()

print("Total ML samples:", len(features_df))
print(
    "Samples with missing features:",
    features_df[feature_columns].isna().any(axis=1).sum()
)

display(missing_counts[missing_counts > 0].to_frame("Missing Values"))

# Identify affected sessions
display(
    features_df.loc[
        features_df[feature_columns].isna().any(axis=1),
        ["Task", "Participant", "Session", "Fatigue_Level"]
    ]
)

Total ML samples: 334
Samples with missing features: 5


,Missing Values
HR_mean,5
HR_median,5
HR_std,5


,Task,Participant,Session,Fatigue_Level
195,Wire Harness,P14,Rep2,High
197,Wire Harness,P14,Rep4,High
208,Wire Harness,P16,Rep5,High
276,Wire Harness,P37,Rep3,High
277,Wire Harness,P37,Rep4,High



## Save Feature Dataset


In [8]:

# Create the features directory
output_dir = project_root / "data" / "features"
output_dir.mkdir(parents=True, exist_ok=True)

# Define output file
output_file = output_dir / "session_features.csv"

# Save all features, identifiers, and fatigue labels
features_df.to_csv(output_file, index=False)

# Verify the saved dataset
print("Saved to:", output_file)
print("Total ML samples:", len(features_df))
print("Sensor features:", len(feature_columns))
print("Total columns:", len(features_df.columns))

print(
    "Samples with missing features:",
    features_df[feature_columns].isna().any(axis=1).sum()
)


Saved to: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\data\features\session_features.csv
Total ML samples: 334
Sensor features: 15
Total columns: 20
Samples with missing features: 5
